# Music Annotation Project
## Notebook 89 — A5 Multi-Threshold High-Resolution Visual Audit

**CPU runtime.** This notebook reads a completed Notebook 88 run. It does not retrain, refit or recalibrate A5.

Outputs use **green = annotation correctly cleaned**, **red = genuine print lost**, and **purple = truth uncertain**. Exact masks and enlarged review overlays are saved separately.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_RUN_OVERRIDE=None
N88_ROOT=ROOT/'a5_contextual_safe_cleanup_v88'
OUTPUT_ROOT=ROOT/'notebook_89_a5_visual_audit'
THRESHOLDS=[0.995,0.990,0.980,0.950]
MIN_CLEAN_RUN=2
REVIEW_RADIUS=3
OVERLAY_ALPHA=.72
CROP_PADDING=120
MIN_CROP_SIDE=520
MAX_CROP_SIDE=1400
MAX_QUEUE=50

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook89_a5_visual_audit'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; CROPS=OUT/'crops'; WHOLE=OUT/'whole_pages'; VISUAL=OUT/'visual_audit'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,CROPS,WHOLE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
    if not x: raise RuntimeError(msg)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def load_gray(path):
    path = Path(path)
    if path.suffix.lower() == '.npy':
        image = np.load(path,allow_pickle=False,)
        require(image.ndim == 2,
            (
                f"{path}: expected 2D cached array, "
                f"got shape {image.shape}"
            ),
        )
        if image.dtype == np.bool_:
            image = (image.astype(np.uint8) * 255)
        elif image.dtype != np.uint8:
            require(
                np.isfinite(image).all(),
                (
                    f"{path}: non-finite image values"
                ),
            )
            require(
                image.min() >= 0
                and image.max() <= 255,
                (
                    f"{path}: grayscale values outside "
                    "[0,255]"
                ),
            )
            image = image.astype(np.uint8)
        return image

    image = cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)

    require(image is not None,
    (
            f"Could not read image: {path}"
        ),
    )
    return image

print('Output:',OUT)


Output: /content/drive/MyDrive/MusicAnnotationProject/notebook_89_a5_visual_audit/20261123_095620_notebook89_a5_visual_audit


## 1. Resolve and validate the completed Notebook 88 run


In [ ]:
PRED_NAMES=['a5_primary_oof_predictions.parquet','all_eligible_oof_predictions.parquet','a5_all_eligible_oof_predictions.parquet','oof_predictions.parquet']

def candidates(root):
    hits=[]
    for name in PRED_NAMES:
        for f in root.rglob(name):
            hits.extend([f.parent,f.parent.parent])
    out=[]; seen=set()
    for p in hits:
        if str(p) in seen: continue
        seen.add(str(p))
        if any((p/'tables'/n).exists() or (p/n).exists() for n in PRED_NAMES): out.append(p)
    return sorted(out,key=lambda p:p.stat().st_mtime)

if SOURCE_RUN_OVERRIDE:
    SOURCE=Path(SOURCE_RUN_OVERRIDE)
else:
    found=candidates(N88_ROOT)
    require(found,'No completed Notebook 88 run found. Set SOURCE_RUN_OVERRIDE to the exact run directory.')
    SOURCE=found[-1]

def find_file(names):
    for n in names:
        for p in [SOURCE/'tables'/n,SOURCE/n]:
            if p.exists(): return p
    return None
PRED=find_file(PRED_NAMES)
require(PRED is not None,'Notebook 88 prediction parquet not found.')
full=pd.read_parquet(PRED)

aliases={
    'page':['page','page_id','stem'], 'source_group':['source_group','group'],
    'y':['y','row','pixel_y'], 'x':['x','col','pixel_x'],
    'truth':['truth','label','truth_class'],
    'prob':['p_annotation_a5','p_clean','clean_probability','prob_clean','calibrated_clean_probability','p_annotation','annotation_probability']}
resolved={k:next((x for x in v if x in full.columns),None) for k,v in aliases.items()}
require(all(resolved[k] for k in ['page','y','x','truth','prob']),f'Could not resolve prediction schema. Columns: {list(full.columns)}')
full=full.rename(columns={v:k for k,v in resolved.items() if v and v!=k}).copy()
if 'source_group' not in full: full['source_group']='UNKNOWN'
for x in ['y','x','truth']: full[x]=pd.to_numeric(full[x],errors='raise').astype(int)
full['prob']=pd.to_numeric(full.prob,errors='raise').astype(float)
full['page']=full.page.astype(str); full['source_group']=full.source_group.astype(str)
require(np.isfinite(full.prob).all(),'Non-finite probabilities found.')
require(not full.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows found.')
require(set(full.truth.unique()).issubset({0,1,2}),'Expected truth labels 0=ANNOTATION, 1=PRINT, 2=UNCERTAIN.')
(AUDIT/'schema_resolution.json').write_text(json.dumps({'source':str(SOURCE),'predictions':str(PRED),'resolved':resolved,'rows':len(full)},indent=2))
print('Source:',SOURCE,'Rows:',len(full),'Pages:',full.page.nunique())


Source: /content/drive/MyDrive/MusicAnnotationProject/a5_contextual_safe_cleanup_v88/20261122_155756_notebook88_a5_training Rows: 324592 Pages: 56


## 2. Locate observed, clean-truth and A1 images


In [ ]:
SUP = {
    '.png',
    '.jpg',
    '.jpeg',
    '.tif',
    '.tiff',
    '.bmp',
}


def imap(folder):

    folder = Path(
        folder
    )

    return {
        p.stem: p

        for p
        in folder.iterdir()

        if (
            p.is_file()
            and p.suffix.lower()
            in SUP
        )
    } if folder.exists() else {}


obsmap = imap(
    ROOT
    / 'aligned'
)

cleanmap = imap(
    ROOT
    / 'source'
    / 'clean'
)


# ------------------------------------------------------------
# Resolve Notebook 88 observable-channel cache
# ------------------------------------------------------------

CHANNEL_CACHE_CANDIDATES = [
    SOURCE
    / 'observable_channel_cache',

    SOURCE
    / 'channel_cache',

    SOURCE
    / 'cache'
    / 'observable_channels',

    SOURCE
    / 'cache',
]


CHANNEL_CACHE = next(
    (
        p

        for p
        in CHANNEL_CACHE_CANDIDATES

        if p.exists()
        and p.is_dir()
    ),
    None,
)


require(
    CHANNEL_CACHE is not None,
    (
        "Could not locate Notebook 88 channel cache. "
        "Tried:\n"
        + "\n".join(
            str(p)

            for p
            in CHANNEL_CACHE_CANDIDATES
        )
    ),
)


print(
    'Notebook 88 channel cache:',
    CHANNEL_CACHE
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def nested(
    page,
    names,
):

    roots = [
        SOURCE
        / 'whole_pages'
        / page,

        SOURCE
        / 'pages'
        / page,

        SOURCE
        / page,
    ]

    for root in roots:

        for name in names:

            path = (
                root
                / name
            )

            if path.exists():

                return path

    return None


def cached_channel(
    page,
    names,
):

    roots = [
        CHANNEL_CACHE
        / page,

        CHANNEL_CACHE,
    ]

    candidates = []

    for root in roots:

        for name in names:

            candidates.extend(
                [
                    root
                    / f'{name}.npy',

                    root
                    / f'{page}_{name}.npy',
                ]
            )

    for path in candidates:

        if path.exists():

            return path

    return None


# ------------------------------------------------------------
# Resolve all page image/array sources
# ------------------------------------------------------------

files = {}
missing = []


for page in sorted(
    full.page.unique()
):

    page = str(
        page
    )


    # --------------------------------------------------------
    # OBSERVED
    #
    # Prefer the exact Notebook 88 cached observed array.
    # Fall back to a rendered Notebook 88 image, then ROOT/aligned.
    # --------------------------------------------------------

    observed = cached_channel(
        page,
        [
            'observed',
            'obs',
        ],
    )

    if observed is None:

        observed = nested(
            page,
            [
                'observed.png',
                'observed_page.png',
            ],
        )

    if observed is None:

        observed = obsmap.get(
            page
        )


    # --------------------------------------------------------
    # CLEAN TRUTH
    #
    # This does not need to be in Notebook 88's output directory.
    # The development clean reference already lives in source/clean.
    # --------------------------------------------------------

    clean = nested(
        page,
        [
            'clean.png',
            'clean_truth.png',
        ],
    )

    if clean is None:

        clean = cleanmap.get(
            page
        )


    # --------------------------------------------------------
    # A1 FALLBACK
    #
    # Prefer the exact cached A1 source used by Notebook 88.
    #
    # Search several plausible durable names because this is
    # an audit notebook and should resolve the actual stored
    # source rather than silently manufacture a new A1.
    # --------------------------------------------------------

    a1 = cached_channel(
        page,
        [
            'a1',
            'a1_source',
            'a1_fallback',
            'baseline',
            'corrected',
        ],
    )

    if a1 is None:

        a1 = nested(
            page,
            [
                'a1.png',
                'a1_fallback.png',
                'baseline.png',
                'v7_8.png',
            ],
        )


    # --------------------------------------------------------
    # Record resolution
    # --------------------------------------------------------

    if (
        observed is None
        or clean is None
        or a1 is None
    ):

        missing.append(
            {
                'page':
                    page,

                'observed':
                    str(
                        observed
                    )
                    if observed
                    else None,

                'clean':
                    str(
                        clean
                    )
                    if clean
                    else None,

                'a1':
                    str(
                        a1
                    )
                    if a1
                    else None,
            }
        )

        continue


    files[
        page
    ] = {
        'observed':
            Path(
                observed
            ),

        'clean':
            Path(
                clean
            ),

        'a1':
            Path(
                a1
            ),
    }


# ------------------------------------------------------------
# Resolution audit
# ------------------------------------------------------------

missing_df = pd.DataFrame(
    missing
)


missing_df.to_csv(
    AUDIT
    / 'missing_images.csv',
    index=False,
)


resolution_rows = []


for page, resolved_files in files.items():

    resolution_rows.append(
        {
            'page':
                page,

            'observed':
                str(
                    resolved_files[
                        'observed'
                    ]
                ),

            'clean':
                str(
                    resolved_files[
                        'clean'
                    ]
                ),

            'a1':
                str(
                    resolved_files[
                        'a1'
                    ]
                ),
        }
    )


pd.DataFrame(
    resolution_rows
).to_csv(
    AUDIT
    / 'image_resolution.csv',
    index=False,
)


require(
    not missing,
    (
        f"{len(missing)} pages still lack one or more "
        "required observed / clean / A1 sources. "
        f"See {AUDIT / 'missing_images.csv'}."
    ),
)


print(
    'Image sets:',
    len(
        files
    )
)

print(
    'Observed:',
    len(
        files
    )
)

print(
    'Clean:',
    len(
        files
    )
)

print(
    'A1:',
    len(
        files
    )
)



RuntimeError: 56 pages lack observed, clean or A1 images. See /content/drive/MyDrive/MusicAnnotationProject/notebook_89_a5_visual_audit/20261123_095620_notebook89_a5_visual_audit/audit/missing_images.csv.

## 3. Recreate threshold decisions and score them


In [ ]:
def run_filter(frame,selected):
    final=np.asarray(selected,bool).copy()
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); im=load_gray(files[page]['observed']); mask=np.zeros(im.shape,np.uint8)
        take=idx[final[idx]]
        if not len(take): continue
        mask[frame.loc[take,'y'].to_numpy(),frame.loc[take,'x'].to_numpy()]=1
        _,lab,stats,_=cv2.connectedComponentsWithStats(mask,8)
        ids=lab[frame.loc[take,'y'].to_numpy(),frame.loc[take,'x'].to_numpy()]
        final[take[stats[ids,cv2.CC_STAT_AREA]<MIN_CLEAN_RUN]]=False
    return final
for t in THRESHOLDS: full[f'sel_{t:.3f}']=run_filter(full,full.prob.to_numpy()>=t)
full['first_threshold']=np.nan
for t in THRESHOLDS: full.loc[full[f'sel_{t:.3f}']&full.first_threshold.isna(),'first_threshold']=t

# Components are connected eligible regions, providing a stable crop identity across thresholds.
compmaps={}; statsmap={}
for page,g in full.groupby('page'):
    im=load_gray(files[page]['observed'])
    m=np.zeros(im.shape,np.uint8); 
    m[g.y,g.x]=1
    _,lab,stats,_=cv2.connectedComponentsWithStats(m,8); compmaps[page]=lab; statsmap[page]=stats

S=[]; P=[]; C=[]
for t in THRESHOLDS:
    col=f'sel_{t:.3f}'; s=full[col]
    S.append({'threshold':t,'selected':int(s.sum()),'annotation_cleaned':int((s&(full.truth==0)).sum()),'print_lost':int((s&(full.truth==1)).sum()),'truth_uncertain':int((s&(full.truth==2)).sum()),'coverage':float(s.mean())})
    for page,g in full.groupby('page'):
        q=g[g[col]].copy(); good=int((q.truth==0).sum()); bad=int((q.truth==1).sum()); unc=int((q.truth==2).sum())
        P.append({'threshold':t,'page':page,'source_group':g.source_group.iloc[0],'selected':len(q),'annotation_cleaned':good,'print_lost':bad,'truth_uncertain':unc,'utility':good-5*bad})
        if q.empty: continue
        q['component_id']=compmaps[page][q.y,q.x]
        for cid,z in q[q.component_id>0].groupby('component_id'):
            good=int((z.truth==0).sum()); bad=int((z.truth==1).sum()); unc=int((z.truth==2).sum()); cat=bad>=50 or (bad/max(good,1)>1 and bad>=20); st=statsmap[page][int(cid)]
            C.append({'threshold':t,'page':page,'source_group':z.source_group.iloc[0],'component_id':int(cid),'selected':len(z),'annotation_cleaned':good,'print_lost':bad,'truth_uncertain':unc,'catastrophic':cat,'utility':good-5*bad-(500 if cat else 0),'x':int(st[0]),'y':int(st[1]),'w':int(st[2]),'h':int(st[3])})
summary=pd.DataFrame(S); summary['print_loss_per_annotation_cleaned']=summary.print_lost/np.maximum(summary.annotation_cleaned,1)
pages=pd.DataFrame(P); components=pd.DataFrame(C)
summary.to_csv(TABLES/'threshold_summary.csv',index=False); pages.to_csv(TABLES/'results_by_page.csv',index=False); components.to_csv(TABLES/'results_by_component.csv',index=False)
full.to_parquet(TABLES/'threshold_decisions.parquet',index=False)
display(summary)


## 4. Generate high-resolution whole-page and component visualisations


In [ ]:
# ============================================================
# 4. High-resolution whole-page and component visual audit
#
# MEMORY-BOUNDED VERSION
#
# Notebook 88 truth semantics:
#
#     0 = PRINT_KEEP
#     1 = ANNOTATION_CLEAN
#     2 = UNCERTAIN
#
# Review colours:
#
#     GREEN  = annotation correctly cleaned
#     RED    = genuine print lost
#     PURPLE = truth uncertain
#
# Memory strategy:
#
#   - process one page at a time;
#   - save full-resolution individual artefacts immediately;
#   - use bounded previews in Matplotlib whole-page grids;
#   - build component overlays directly in crop space;
#   - never retain full-page crop views;
#   - explicitly clear Matplotlib Axes and Figures;
#   - release page arrays before loading the next page.
# ============================================================

import gc
import json
import math

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# Resolve the Notebook 87 run containing registered clean truth
# ============================================================

if "SOURCE_N87" not in globals():

    identity_path = (
        SOURCE
        / "experiment_identity.json"
    )

    require(
        identity_path.exists(),
        (
            "Missing Notebook 88 experiment identity: "
            f"{identity_path}"
        ),
    )

    n88_identity = json.loads(
        identity_path.read_text()
    )

    require(
        "source_notebook87_run"
        in n88_identity,
        (
            "Notebook 88 experiment identity does not contain "
            "'source_notebook87_run'."
        ),
    )

    SOURCE_N87 = Path(
        n88_identity[
            "source_notebook87_run"
        ]
    )

    require(
        SOURCE_N87.exists(),
        (
            "Notebook 87 source run does not exist: "
            f"{SOURCE_N87}"
        ),
    )


# ============================================================
# Visual controls
# ============================================================

GREEN = (
    35,
    190,
    70,
)

RED = (
    20,
    20,
    240,
)

PURPLE = (
    210,
    40,
    210,
)


TC = {
    0.995: (
        220,
        70,
        20,
    ),

    0.990: (
        255,
        190,
        0,
    ),

    0.980: (
        0,
        215,
        255,
    ),

    0.950: (
        0,
        110,
        255,
    ),
}


# Full-resolution individual PNGs are preserved.
#
# This limit applies only to images inserted into Matplotlib
# whole-page review grids.
GRID_PREVIEW_MAX_SIDE = 1400


# ============================================================
# Path resolvers
# ============================================================

def resolve_registered_clean_path(page):

    page = str(
        page
    )

    candidates = [
        (
            SOURCE_N87
            / "truth_registration"
            / "registered_clean"
            / f"{page}_clean_registered.png"
        ),

        (
            SOURCE
            / "truth_registration"
            / "registered_clean"
            / f"{page}_clean_registered.png"
        ),
    ]

    for path in candidates:

        if path.exists():

            return path

    raise RuntimeError(
        (
            f"{page}: registered clean truth not found.\n"
            "Notebook 89 must not use the original "
            "unregistered source/clean image for pixelwise "
            "comparison.\nTried:\n"
            + "\n".join(
                str(path)
                for path in candidates
            )
        )
    )


def resolve_paper_path(page):

    page = str(
        page
    )

    paper = files[
        page
    ].get(
        "paper"
    )

    if (
        paper is not None
        and Path(
            paper
        ).exists()
    ):

        return Path(
            paper
        )

    a1_path = Path(
        files[
            page
        ][
            "a1"
        ]
    )

    sibling = (
        a1_path.parent
        / "paper.npy"
    )

    if sibling.exists():

        return sibling

    if (
        "CHANNEL_CACHE"
        in globals()
        and CHANNEL_CACHE is not None
    ):

        candidate_path = (
            Path(
                CHANNEL_CACHE
            )
            / page
            / "paper.npy"
        )

        if candidate_path.exists():

            return candidate_path

    raise RuntimeError(
        (
            f"{page}: could not resolve the frozen "
            "Notebook 88 paper estimate. Expected paper.npy "
            "beside the cached A1 array or under "
            "CHANNEL_CACHE/<page>/paper.npy."
        )
    )


# ============================================================
# Bounded preview helper
# ============================================================

def grid_preview(
    image,
    maximum_side=GRID_PREVIEW_MAX_SIDE,
):

    image = np.asarray(
        image
    )

    require(
        image.ndim
        in (
            2,
            3,
        ),
        (
            "Grid preview requires a 2D grayscale or "
            f"3D colour array; got {image.shape}"
        ),
    )

    height, width = image.shape[
        :2
    ]

    largest_side = max(
        height,
        width,
    )

    if largest_side <= maximum_side:

        return image.copy()

    scale = (
        float(
            maximum_side
        )
        / float(
            largest_side
        )
    )

    output_width = max(
        1,
        int(
            round(
                width
                * scale
            )
        ),
    )

    output_height = max(
        1,
        int(
            round(
                height
                * scale
            )
        ),
    )

    return cv2.resize(
        image,
        (
            output_width,
            output_height,
        ),
        interpolation=cv2.INTER_AREA,
    )


# ============================================================
# Mask dilation
# ============================================================

def dil(
    mask,
    radius=REVIEW_RADIUS,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    if not radius:

        return mask.copy()

    kernel = np.ones(
        (
            2 * radius + 1,
            2 * radius + 1,
        ),
        dtype=np.uint8,
    )

    result = (
        cv2.dilate(
            mask.astype(
                np.uint8
            ),
            kernel,
        )
        > 0
    )

    del kernel

    return result


# ============================================================
# Overlay helper
# ============================================================

def overlay(
    gray,
    layers,
):

    require(
        gray.ndim == 2,
        (
            "Overlay base must be a 2D grayscale image; "
            f"got {gray.shape}"
        ),
    )

    base = cv2.cvtColor(
        gray,
        cv2.COLOR_GRAY2BGR,
    )

    colour = base.copy()

    active = np.zeros(
        gray.shape,
        dtype=bool,
    )

    for mask, bgr in layers:

        mask = np.asarray(
            mask,
            dtype=bool,
        )

        require(
            mask.shape
            == gray.shape,
            (
                "Overlay mask shape mismatch: "
                f"{mask.shape} != {gray.shape}"
            ),
        )

        colour[
            mask
        ] = bgr

        active |= mask

    if active.any():

        mixed = cv2.addWeighted(
            base,
            1.0
            - OVERLAY_ALPHA,
            colour,
            OVERLAY_ALPHA,
            0,
        )

        base[
            active
        ] = mixed[
            active
        ]

        del mixed

    del colour
    del active

    return base


# ============================================================
# Memory-bounded Matplotlib grid writer
# ============================================================

def savegrid(
    images,
    titles,
    path,
    title,
    cols=4,
    dpi=145,
):

    require(
        len(
            images
        )
        == len(
            titles
        ),
        (
            "Image/title count mismatch: "
            f"{len(images)} images versus "
            f"{len(titles)} titles"
        ),
    )

    require(
        len(
            images
        )
        > 0,
        "Cannot create an empty review grid.",
    )

    rows = math.ceil(
        len(
            images
        )
        / cols
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            4.2 * cols,
            4.2 * rows,
        ),
        squeeze=False,
    )

    try:

        for axis in axes.flat:

            axis.set_axis_off()

        for axis, image, panel_title in zip(
            axes.flat,
            images,
            titles,
        ):

            require(
                image is not None,
                (
                    f"Panel {panel_title!r} "
                    "contains no image."
                ),
            )

            if image.ndim == 3:

                display_image = cv2.cvtColor(
                    image,
                    cv2.COLOR_BGR2RGB,
                )

                axis.imshow(
                    display_image,
                    interpolation="nearest",
                )

                del display_image

            else:

                axis.imshow(
                    image,
                    cmap="gray",
                    vmin=0,
                    vmax=255,
                    interpolation="nearest",
                )

            axis.set_title(
                panel_title,
                fontsize=9,
            )

            axis.set_axis_off()

        fig.suptitle(
            title,
            fontsize=12,
        )

        # Avoid tight_layout/bbox_inches="tight". Both can force
        # additional large renderer passes.
        fig.subplots_adjust(
            left=0.01,
            right=0.99,
            bottom=0.01,
            top=0.96,
            wspace=0.04,
            hspace=0.12,
        )

        fig.savefig(
            path,
            dpi=dpi,
            bbox_inches=None,
            pad_inches=0.05,
        )

    finally:

        # Remove AxesImage references before closing the figure.
        for axis in axes.flat:

            axis.clear()

        fig.clear()

        plt.close(
            fig
        )

        del axes
        del fig

        gc.collect()


# ============================================================
# Truth masks for one threshold
# ============================================================

def masks(
    page,
    group,
    threshold,
    shape,
    region=None,
):

    selected = (
        group[
            f"sel_{threshold:.3f}"
        ].to_numpy(
            dtype=bool,
            copy=False,
        )
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    truth = group.truth.to_numpy(
        dtype=np.int8,
        copy=False,
    )

    good = np.zeros(
        shape,
        dtype=bool,
    )

    bad = np.zeros(
        shape,
        dtype=bool,
    )

    uncertain = np.zeros(
        shape,
        dtype=bool,
    )

    # 1 = ANNOTATION_CLEAN
    good_hit = (
        selected
        & (
            truth == 1
        )
    )

    good[
        ys[
            good_hit
        ],
        xs[
            good_hit
        ],
    ] = True

    # 0 = PRINT_KEEP
    bad_hit = (
        selected
        & (
            truth == 0
        )
    )

    bad[
        ys[
            bad_hit
        ],
        xs[
            bad_hit
        ],
    ] = True

    # 2 = UNCERTAIN
    uncertain_hit = (
        selected
        & (
            truth == 2
        )
    )

    uncertain[
        ys[
            uncertain_hit
        ],
        xs[
            uncertain_hit
        ],
    ] = True

    if region is not None:

        require(
            region.shape
            == shape,
            (
                f"{page}: component-region shape "
                f"{region.shape} != page shape {shape}"
            ),
        )

        good &= region
        bad &= region
        uncertain &= region

    return (
        good,
        bad,
        uncertain,
    )


# ============================================================
# Whole-page A5 candidate assembly
#
# Start from exact A1 and replace CLEAN-selected pixels with the
# frozen paper estimate.
# ============================================================

def candidate(
    group,
    threshold,
    a1,
    paper,
):

    require(
        a1.shape
        == paper.shape,
        (
            "A1/paper shape mismatch: "
            f"{a1.shape} != {paper.shape}"
        ),
    )

    output = a1.copy()

    selected = (
        group[
            f"sel_{threshold:.3f}"
        ].to_numpy(
            dtype=bool,
            copy=False,
        )
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    output[
        ys[
            selected
        ],
        xs[
            selected
        ],
    ] = paper[
        ys[
            selected
        ],
        xs[
            selected
        ],
    ]

    return output


# ============================================================
# Crop-local candidate assembly
#
# This avoids constructing a full-page candidate for every
# component and threshold.
# ============================================================

def candidate_crop(
    group,
    threshold,
    a1_crop,
    paper_crop,
    x0,
    y0,
    x1,
    y1,
):

    output = a1_crop.copy()

    selected = (
        group[
            f"sel_{threshold:.3f}"
        ].to_numpy(
            dtype=bool,
            copy=False,
        )
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    inside = (
        selected
        & (
            ys >= y0
        )
        & (
            ys < y1
        )
        & (
            xs >= x0
        )
        & (
            xs < x1
        )
    )

    crop_y = (
        ys[
            inside
        ]
        - y0
    )

    crop_x = (
        xs[
            inside
        ]
        - x0
    )

    output[
        crop_y,
        crop_x,
    ] = paper_crop[
        crop_y,
        crop_x,
    ]

    return output


# ============================================================
# Crop geometry
# ============================================================

def cropbox(
    shape,
    x,
    y,
    width,
    height,
):

    image_height, image_width = shape

    side = min(
        MAX_CROP_SIDE,
        max(
            MIN_CROP_SIDE,
            width
            + 2
            * CROP_PADDING,
            height
            + 2
            * CROP_PADDING,
        ),
    )

    centre_x = (
        x
        + width
        // 2
    )

    centre_y = (
        y
        + height
        // 2
    )

    x0 = max(
        0,
        min(
            max(
                image_width
                - side,
                0,
            ),
            centre_x
            - side
            // 2,
        ),
    )

    y0 = max(
        0,
        min(
            max(
                image_height
                - side,
                0,
            ),
            centre_y
            - side
            // 2,
        ),
    )

    return (
        x0,
        y0,
        min(
            image_width,
            x0
            + side,
        ),
        min(
            image_height,
            y0
            + side,
        ),
    )


# ============================================================
# Whole-page comparisons
#
# Full-resolution individual outputs are written immediately.
# Only reduced previews are kept for the whole-page grid.
# ============================================================

page_groups = full.groupby(
    "page",
    sort=False,
)

number_of_pages = int(
    full.page.nunique()
)


for page_number, (
    page,
    group_source,
) in enumerate(
    page_groups,
    start=1,
):

    page = str(
        page
    )

    group = group_source.reset_index(
        drop=True
    )

    observed = load_gray(
        files[
            page
        ][
            "observed"
        ]
    )

    clean_path = resolve_registered_clean_path(
        page
    )

    clean = load_gray(
        clean_path
    )

    a1 = load_gray(
        files[
            page
        ][
            "a1"
        ]
    )

    paper_path = resolve_paper_path(
        page
    )

    paper = load_gray(
        paper_path
    )

    require(
        (
            observed.shape
            == clean.shape
            == a1.shape
            == paper.shape
        ),
        (
            f"{page}: observed/registered-clean/A1/paper "
            f"shape mismatch: "
            f"observed={observed.shape}, "
            f"clean={clean.shape}, "
            f"A1={a1.shape}, "
            f"paper={paper.shape}"
        ),
    )

    output_dir = (
        WHOLE
        / page
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    require(
        (
            (
                ys >= 0
            ).all()
            and (
                xs >= 0
            ).all()
            and (
                ys
                < observed.shape[
                    0
                ]
            ).all()
            and (
                xs
                < observed.shape[
                    1
                ]
            ).all()
        ),
        (
            f"{page}: prediction coordinate outside "
            "the resolved page arrays."
        ),
    )

    first_threshold = (
        group.first_threshold.to_numpy(
            dtype=np.float64,
            copy=False,
        )
    )

    transition_layers = []

    for threshold in THRESHOLDS:

        threshold_hit = np.isclose(
            first_threshold,
            threshold,
            rtol=0.0,
            atol=1e-12,
            equal_nan=False,
        )

        transition_mask = np.zeros(
            observed.shape,
            dtype=bool,
        )

        transition_mask[
            ys[
                threshold_hit
            ],
            xs[
                threshold_hit
            ],
        ] = True

        review_transition_mask = dil(
            transition_mask
        )

        transition_layers.append(
            (
                review_transition_mask,
                TC[
                    threshold
                ],
            )
        )

        del threshold_hit
        del transition_mask
        del review_transition_mask

    transition_overlay = overlay(
        observed,
        transition_layers,
    )

    # Only bounded previews enter the grid.
    images = [
        grid_preview(
            observed
        ),

        grid_preview(
            clean
        ),

        grid_preview(
            a1
        ),

        grid_preview(
            transition_overlay
        ),
    ]

    titles = [
        "Observed annotated",
        "Registered clean truth",
        "A1",
        "First threshold selected",
    ]

    for threshold in THRESHOLDS:

        (
            good,
            bad,
            uncertain,
        ) = masks(
            page,
            group,
            threshold,
            observed.shape,
        )

        candidate_image = candidate(
            group,
            threshold,
            a1,
            paper,
        )

        exact_overlay = overlay(
            observed,
            [
                (
                    good,
                    GREEN,
                ),
                (
                    bad,
                    RED,
                ),
                (
                    uncertain,
                    PURPLE,
                ),
            ],
        )

        good_review = dil(
            good
        )

        bad_review = dil(
            bad
        )

        uncertain_review = dil(
            uncertain
        )

        review_overlay = overlay(
            observed,
            [
                (
                    good_review,
                    GREEN,
                ),
                (
                    bad_review,
                    RED,
                ),
                (
                    uncertain_review,
                    PURPLE,
                ),
            ],
        )

        require(
            cv2.imwrite(
                str(
                    output_dir
                    / f"a5_{threshold:.3f}.png"
                ),
                candidate_image,
            ),
            (
                f"{page}: failed to write A5 candidate "
                f"at threshold {threshold:.3f}."
            ),
        )

        require(
            cv2.imwrite(
                str(
                    output_dir
                    / f"truth_exact_{threshold:.3f}.png"
                ),
                exact_overlay,
            ),
            (
                f"{page}: failed to write exact overlay "
                f"at threshold {threshold:.3f}."
            ),
        )

        require(
            cv2.imwrite(
                str(
                    output_dir
                    / f"truth_review_{threshold:.3f}.png"
                ),
                review_overlay,
            ),
            (
                f"{page}: failed to write review overlay "
                f"at threshold {threshold:.3f}."
            ),
        )

        # Retain only reduced copies for the grid.
        images.extend(
            [
                grid_preview(
                    candidate_image
                ),

                grid_preview(
                    review_overlay
                ),
            ]
        )

        titles.extend(
            [
                f"A5 @ {threshold:.3f}",
                (
                    "Green cleanup / red print loss "
                    f"@ {threshold:.3f}"
                ),
            ]
        )

        # Full-resolution threshold objects are no longer needed.
        del good
        del bad
        del uncertain
        del candidate_image
        del exact_overlay
        del good_review
        del bad_review
        del uncertain_review
        del review_overlay

        gc.collect()

    savegrid(
        images,
        titles,
        output_dir
        / "whole_page_review.png",
        f"{page} - A5 thresholds",
        cols=4,
        dpi=145,
    )

    images.clear()
    titles.clear()
    transition_layers.clear()

    del images
    del titles
    del transition_layers
    del transition_overlay
    del first_threshold
    del ys
    del xs
    del observed
    del clean
    del clean_path
    del a1
    del paper
    del paper_path
    del group
    del group_source

    plt.close(
        "all"
    )

    gc.collect()

    if (
        page_number % 5 == 0
        or page_number == number_of_pages
    ):

        print(
            "Whole pages:",
            page_number,
            "/",
            number_of_pages,
        )


# ============================================================
# Component roster
#
# PAGE-GROUPED, MATPLOTLIB-FREE COMPONENT RENDERER
#
# Each page is loaded exactly once for all of its components.
# Component sheets are assembled with OpenCV rather than
# Matplotlib, avoiding thousands of retained renderer buffers.
# ============================================================

union = (
    components
    .groupby(
        [
            "page",
            "source_group",
            "component_id",
        ],
        as_index=False,
    )
    .agg(
        max_annotation_cleaned=(
            "annotation_cleaned",
            "max",
        ),

        max_print_lost=(
            "print_lost",
            "max",
        ),

        max_uncertain=(
            "truth_uncertain",
            "max",
        ),

        ever_catastrophic=(
            "catastrophic",
            "max",
        ),

        x=(
            "x",
            "first",
        ),

        y=(
            "y",
            "first",
        ),

        w=(
            "w",
            "first",
        ),

        h=(
            "h",
            "first",
        ),
    )
)


union[
    "page"
] = union[
    "page"
].astype(
    str
)


# Precompute this once. Do not repeatedly execute:
#
#     full.page.astype(str) == page
#
# for every component.
full_by_page = {
    str(page):
        group.reset_index(
            drop=True
        )

    for page, group
    in full.groupby(
        "page",
        sort=False,
    )
}


# ============================================================
# OpenCV component-sheet helpers
# ============================================================

COMPONENT_PANEL_WIDTH = 520
COMPONENT_PANEL_HEIGHT = 520
COMPONENT_PANEL_HEADER = 42

COMPONENT_GRID_COLUMNS = 4

COMPONENT_SHEET_TITLE_HEIGHT = 76

COMPONENT_SHEET_JPEG_QUALITY = 94


def fit_panel(
    image,
    panel_width=COMPONENT_PANEL_WIDTH,
    panel_height=COMPONENT_PANEL_HEIGHT,
):

    """
    Place a grayscale or BGR image into a fixed-size white panel
    without changing its aspect ratio.
    """

    require(
        image is not None,
        "Component panel image is None.",
    )

    require(
        image.ndim in (
            2,
            3,
        ),
        (
            "Component panel must be 2D grayscale "
            f"or 3D colour; got {image.shape}"
        ),
    )

    if image.ndim == 2:

        image_bgr = cv2.cvtColor(
            image,
            cv2.COLOR_GRAY2BGR,
        )

    else:

        image_bgr = image


    source_height, source_width = image_bgr.shape[
        :2
    ]


    require(
        source_height > 0
        and source_width > 0,
        (
            "Component panel has an empty source image: "
            f"{image_bgr.shape}"
        ),
    )


    available_height = (
        panel_height
        - COMPONENT_PANEL_HEADER
    )


    scale = min(
        panel_width
        / source_width,
        available_height
        / source_height,
    )


    output_width = max(
        1,
        int(
            round(
                source_width
                * scale
            )
        ),
    )


    output_height = max(
        1,
        int(
            round(
                source_height
                * scale
            )
        ),
    )


    interpolation = (
        cv2.INTER_AREA

        if scale < 1.0

        else cv2.INTER_NEAREST
    )


    resized = cv2.resize(
        image_bgr,
        (
            output_width,
            output_height,
        ),
        interpolation=interpolation,
    )


    panel = np.full(
        (
            panel_height,
            panel_width,
            3,
        ),
        255,
        dtype=np.uint8,
    )


    x_offset = (
        panel_width
        - output_width
    ) // 2


    y_offset = (
        COMPONENT_PANEL_HEADER
        + (
            available_height
            - output_height
        ) // 2
    )


    panel[
        y_offset:
            y_offset
            + output_height,

        x_offset:
            x_offset
            + output_width,
    ] = resized


    if image_bgr is not image:
        del image_bgr

    del resized

    return panel


def add_panel_title(
    panel,
    title,
):

    scale = 0.55

    maximum_character_width = max(
        1,
        int(
            (
                panel.shape[
                    1
                ]
                - 20
            )
            / 10
        ),
    )


    display_title = str(
        title
    )


    if (
        len(
            display_title
        )
        > maximum_character_width
    ):

        display_title = (
            display_title[
                :
                maximum_character_width
                - 3
            ]
            + "..."
        )


    cv2.putText(
        panel,
        display_title,
        (
            10,
            27,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        (
            30,
            30,
            30,
        ),
        1,
        cv2.LINE_AA,
    )


    return panel


def save_component_grid(
    images,
    titles,
    path,
    sheet_title,
    columns=COMPONENT_GRID_COLUMNS,
):

    require(
        len(
            images
        )
        == len(
            titles
        ),
        (
            "Component image/title mismatch: "
            f"{len(images)} != {len(titles)}"
        ),
    )


    require(
        len(
            images
        )
        > 0,
        "Cannot produce an empty component grid.",
    )


    panel_rows = math.ceil(
        len(
            images
        )
        / columns
    )


    sheet_width = (
        columns
        * COMPONENT_PANEL_WIDTH
    )


    sheet_height = (
        COMPONENT_SHEET_TITLE_HEIGHT
        + panel_rows
        * COMPONENT_PANEL_HEIGHT
    )


    sheet = np.full(
        (
            sheet_height,
            sheet_width,
            3,
        ),
        255,
        dtype=np.uint8,
    )


    # Sheet heading.
    heading = str(
        sheet_title
    )


    if len(
        heading
    ) > 240:

        heading = (
            heading[
                :237
            ]
            + "..."
        )


    cv2.putText(
        sheet,
        heading,
        (
            14,
            34,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.62,
        (
            25,
            25,
            25,
        ),
        1,
        cv2.LINE_AA,
    )


    cv2.putText(
        sheet,
        (
            "Green = annotation cleaned; "
            "red = print lost; purple = uncertain"
        ),
        (
            14,
            61,
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.53,
        (
            50,
            50,
            50,
        ),
        1,
        cv2.LINE_AA,
    )


    for panel_index, (
        image,
        panel_title,
    ) in enumerate(
        zip(
            images,
            titles,
        )
    ):

        panel = fit_panel(
            image
        )

        panel = add_panel_title(
            panel,
            panel_title,
        )


        row_number = (
            panel_index
            // columns
        )


        column_number = (
            panel_index
            % columns
        )


        y0 = (
            COMPONENT_SHEET_TITLE_HEIGHT
            + row_number
            * COMPONENT_PANEL_HEIGHT
        )


        x0 = (
            column_number
            * COMPONENT_PANEL_WIDTH
        )


        sheet[
            y0:
                y0
                + COMPONENT_PANEL_HEIGHT,

            x0:
                x0
                + COMPONENT_PANEL_WIDTH,
        ] = panel


        del panel


    path = Path(
        path
    )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    # PNG is lossless but can be memory-expensive during encoding.
    # JPEG at quality 94 is sufficient for these review sheets.
    jpeg_path = path.with_suffix(
        ".jpg"
    )


    success = cv2.imwrite(
        str(
            jpeg_path
        ),
        sheet,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            COMPONENT_SHEET_JPEG_QUALITY,
        ],
    )


    require(
        success,
        (
            "Could not write component review sheet: "
            f"{jpeg_path}"
        ),
    )


    del sheet

    return jpeg_path


# ============================================================
# Render pages and components
# ============================================================

rows = []

component_counter = 0

component_total = int(
    len(
        union
    )
)


for page_number, (
    page,
    page_components,
) in enumerate(
    union.groupby(
        "page",
        sort=False,
    ),
    start=1,
):

    page = str(
        page
    )


    require(
        page in full_by_page,
        (
            f"{page}: prediction rows absent from "
            "the page-group map."
        ),
    )


    group = full_by_page[
        page
    ]


    # --------------------------------------------------------
    # Load this page exactly once.
    # --------------------------------------------------------

    observed = load_gray(
        files[
            page
        ][
            "observed"
        ]
    )


    clean_path = resolve_registered_clean_path(
        page
    )


    clean = load_gray(
        clean_path
    )


    a1 = load_gray(
        files[
            page
        ][
            "a1"
        ]
    )


    paper_path = resolve_paper_path(
        page
    )


    paper = load_gray(
        paper_path
    )


    require(
        (
            observed.shape
            == clean.shape
            == a1.shape
            == paper.shape
        ),
        (
            f"{page}: observed/registered-clean/A1/paper "
            f"shape mismatch: "
            f"observed={observed.shape}, "
            f"clean={clean.shape}, "
            f"A1={a1.shape}, "
            f"paper={paper.shape}"
        ),
    )


    require(
        page
        in compmaps,
        (
            f"{page}: component map is missing."
        ),
    )


    component_labels = compmaps[
        page
    ]


    require(
        component_labels.shape
        == observed.shape,
        (
            f"{page}: component-label map shape "
            f"{component_labels.shape} != "
            f"{observed.shape}"
        ),
    )


    # Arrays used for every component on this page.
    group_y = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )


    group_x = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )


    group_truth = group.truth.to_numpy(
        dtype=np.int8,
        copy=False,
    )


    threshold_selections = {
        float(
            threshold
        ):
            group[
                f"sel_{threshold:.3f}"
            ].to_numpy(
                dtype=bool,
                copy=False,
            )

        for threshold
        in THRESHOLDS
    }


    for component in page_components.itertuples(
        index=False
    ):

        component_counter += 1


        component_id = int(
            component.component_id
        )


        x0, y0, x1, y1 = cropbox(
            observed.shape,
            int(
                component.x
            ),
            int(
                component.y
            ),
            int(
                component.w
            ),
            int(
                component.h
            ),
        )


        # Source crops are independent of their full-page parents.
        observed_crop = observed[
            y0:y1,
            x0:x1,
        ].copy()


        clean_crop = clean[
            y0:y1,
            x0:x1,
        ].copy()


        a1_crop = a1[
            y0:y1,
            x0:x1,
        ].copy()


        paper_crop = paper[
            y0:y1,
            x0:x1,
        ].copy()


        component_region_crop = (
            component_labels[
                y0:y1,
                x0:x1,
            ]
            == component_id
        )


        require(
            component_region_crop.shape
            == observed_crop.shape,
            (
                f"{page}/component {component_id}: "
                "component crop shape mismatch."
            ),
        )


        crop_height, crop_width = (
            observed_crop.shape
        )


        inside_crop = (
            (
                group_y
                >= y0
            )
            & (
                group_y
                < y1
            )
            & (
                group_x
                >= x0
            )
            & (
                group_x
                < x1
            )
        )


        crop_y = (
            group_y[
                inside_crop
            ]
            - y0
        )


        crop_x = (
            group_x[
                inside_crop
            ]
            - x0
        )


        crop_truth_values = group_truth[
            inside_crop
        ]


        truth_map_crop = np.full(
            (
                crop_height,
                crop_width,
            ),
            -1,
            dtype=np.int8,
        )


        truth_map_crop[
            crop_y,
            crop_x,
        ] = crop_truth_values


        images = [
            observed_crop.copy(),
            clean_crop.copy(),
            a1_crop.copy(),
        ]


        titles = [
            "Observed",
            "Registered clean truth",
            "A1",
        ]


        metrics = []


        previous_crop = np.zeros(
            (
                crop_height,
                crop_width,
            ),
            dtype=bool,
        )


        for threshold in THRESHOLDS:

            selection = threshold_selections[
                float(
                    threshold
                )
            ]


            selected_inside = (
                selection
                & inside_crop
            )


            selected_crop = np.zeros(
                (
                    crop_height,
                    crop_width,
                ),
                dtype=bool,
            )


            crop_selection_y = (
                group_y[
                    selected_inside
                ]
                - y0
            )


            crop_selection_x = (
                group_x[
                    selected_inside
                ]
                - x0
            )


            selected_crop[
                crop_selection_y,
                crop_selection_x,
            ] = True


            # Restrict this visual sheet to the selected
            # authoritative component.
            selected_crop &= component_region_crop


            good_crop = (
                selected_crop
                & (
                    truth_map_crop
                    == 1
                )
            )


            bad_crop = (
                selected_crop
                & (
                    truth_map_crop
                    == 0
                )
            )


            uncertain_crop = (
                selected_crop
                & (
                    truth_map_crop
                    == 2
                )
            )


            # Build this threshold's candidate directly in crop
            # space.
            candidate_crop_image = a1_crop.copy()


            selected_crop_y, selected_crop_x = np.where(
                selected_crop
            )


            candidate_crop_image[
                selected_crop_y,
                selected_crop_x,
            ] = paper_crop[
                selected_crop_y,
                selected_crop_x,
            ]


            truth_view_crop = overlay(
                observed_crop,
                [
                    (
                        dil(
                            good_crop
                        ),
                        GREEN,
                    ),
                    (
                        dil(
                            bad_crop
                        ),
                        RED,
                    ),
                    (
                        dil(
                            uncertain_crop
                        ),
                        PURPLE,
                    ),
                ],
            )


            difference_crop = np.clip(
                (
                    cv2.absdiff(
                        a1_crop,
                        candidate_crop_image,
                    ).astype(
                        np.int16
                    )
                    * 4
                ),
                0,
                255,
            ).astype(
                np.uint8
            )


            incremental_crop = (
                selected_crop
                & ~previous_crop
            )


            incremental_view_crop = overlay(
                observed_crop,
                [
                    (
                        dil(
                            incremental_crop
                            & (
                                truth_map_crop
                                == 1
                            )
                        ),
                        GREEN,
                    ),
                    (
                        dil(
                            incremental_crop
                            & (
                                truth_map_crop
                                == 0
                            )
                        ),
                        RED,
                    ),
                    (
                        dil(
                            incremental_crop
                            & (
                                truth_map_crop
                                == 2
                            )
                        ),
                        PURPLE,
                    ),
                ],
            )


            images.extend(
                [
                    candidate_crop_image,
                    truth_view_crop,
                    difference_crop,
                    incremental_view_crop,
                ]
            )


            titles.extend(
                [
                    f"A5 {threshold:.3f}",
                    f"Truth overlay {threshold:.3f}",
                    f"|A1-A5| x4 {threshold:.3f}",
                    f"NEW at {threshold:.3f}",
                ]
            )


            metrics.append(
                (
                    f"{threshold:.3f}: "
                    f"{int(selected_crop.sum())} selected, "
                    f"{int(good_crop.sum())} good, "
                    f"{int(bad_crop.sum())} lost, "
                    f"{int(uncertain_crop.sum())} uncertain"
                )
            )


            previous_crop = selected_crop.copy()


            del crop_selection_y
            del crop_selection_x
            del selected_crop_y
            del selected_crop_x
            del selected_inside
            del selected_crop
            del good_crop
            del bad_crop
            del uncertain_crop
            del incremental_crop


        component_folder = (
            CROPS
            / page
        )


        component_folder.mkdir(
            parents=True,
            exist_ok=True,
        )


        requested_path = (
            component_folder
            / (
                f"{page}_component_"
                f"{component_id:04d}.png"
            )
        )


        saved_path = save_component_grid(
            images,
            titles,
            requested_path,
            (
                f"{page} component {component_id} | "
                + " | ".join(
                    metrics
                )
            ),
            columns=4,
        )


        rows.append({
            "page":
                page,

            "source_group":
                component.source_group,

            "component_id":
                component_id,

            "crop_path":
                str(
                    saved_path
                ),

            "max_annotation_cleaned":
                int(
                    component.max_annotation_cleaned
                ),

            "max_print_lost":
                int(
                    component.max_print_lost
                ),

            "max_uncertain":
                int(
                    component.max_uncertain
                ),

            "ever_catastrophic":
                bool(
                    component.ever_catastrophic
                ),
        })


        # Component-local cleanup.
        images.clear()
        titles.clear()
        metrics.clear()


        del images
        del titles
        del metrics
        del observed_crop
        del clean_crop
        del a1_crop
        del paper_crop
        del component_region_crop
        del inside_crop
        del crop_y
        del crop_x
        del crop_truth_values
        del truth_map_crop
        del previous_crop
        del candidate_crop_image
        del truth_view_crop
        del difference_crop
        del incremental_view_crop


        if (
            component_counter % 25
            == 0
            or component_counter
            == component_total
        ):

            print(
                "Components:",
                component_counter,
                "/",
                component_total,
                "| page:",
                page,
            )


        gc.collect()


    # --------------------------------------------------------
    # Page-level cleanup occurs only after every component on
    # this page has been written.
    # --------------------------------------------------------

    del observed
    del clean
    del a1
    del paper
    del component_labels
    del group_y
    del group_x
    del group_truth
    del threshold_selections
    del clean_path
    del paper_path
    del group
    del page_components

    gc.collect()


    print(
        "Released page:",
        page,
        "| completed components:",
        component_counter,
        "/",
        component_total,
    )


# ============================================================
# Component crop index
# ============================================================

crop_index = pd.DataFrame(
    rows
)


crop_index.to_csv(
    TABLES
    / "component_crop_index.csv",
    index=False,
)


print()

print(
    "Detailed component crops:",
    len(
        crop_index
    )
)

print(
    "Component sheets:",
    "OpenCV JPEG montage, no Matplotlib"
)

print(
    "Page handling:",
    "one load per page, explicit release after all page components"
)

# ============================================================
# Completion summary
# ============================================================

print()

print(
    "Detailed component crops:",
    len(
        crop_index
    )
)

print(
    "Truth semantics:",
    (
        "green=ANNOTATION_CLEAN, "
        "red=PRINT_KEEP loss, "
        "purple=UNCERTAIN"
    )
)

print(
    "Whole-page grid inputs:",
    (
        f"maximum side {GRID_PREVIEW_MAX_SIDE}px; "
        "individual page PNGs remain full resolution"
    )
)

print(
    "Component rendering:",
    (
        "crop-local overlays and candidate images; "
        "no full-page RGB component overlays"
    )
)

print(
    "Memory handling:",
    (
        "one page at a time, immediate full-resolution writes, "
        "bounded grid previews, copied crops and explicit "
        "Matplotlib cleanup"
    )
)

## 5. Build review queues, contact sheets and upload package


In [ ]:
queues={
 'catastrophic':crop_index[crop_index.ever_catastrophic].sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[False,False]),
 'false_heavy':crop_index.sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[False,False]),
 'beneficial':crop_index[crop_index.max_annotation_cleaned>0].sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[True,False])}
# Threshold-boundary ranking: print loss newly entering at 0.950, plus useful gain by 0.980.
piv=components.pivot_table(index=['page','component_id'],columns='threshold',values=['annotation_cleaned','print_lost'],fill_value=0)
br=[]
for (page,cid),r in piv.iterrows():
    get=lambda metric,t:int(r[(metric,t)]) if (metric,t) in r.index else 0
    br.append({'page':page,'component_id':cid,'safe_gain_to_0980':get('annotation_cleaned',.980)-get('annotation_cleaned',.995),'new_loss_at_0950':get('print_lost',.950)-get('print_lost',.980)})
queues['threshold_boundary']=pd.DataFrame(br).merge(crop_index,on=['page','component_id']).sort_values(['new_loss_at_0950','safe_gain_to_0980'],ascending=[False,False])
for name,q in queues.items():
    q.head(MAX_QUEUE).to_csv(TABLES/f'{name}_review_queue.csv',index=False)
    ims=[]; titles=[]
    for _,r in q.head(24).iterrows():
        im=cv2.imread(str(r.crop_path));
        if im is not None: ims.append(im); titles.append(f'{r.page} C{int(r.component_id)} good={int(r.max_annotation_cleaned)} lost={int(r.max_print_lost)}')
    if ims: savegrid(ims,titles,VISUAL/f'{name}_contact_sheet.png',name.replace('_',' ').title(),4,120)

plt.figure(figsize=(9,6)); plt.plot(summary.print_loss_per_annotation_cleaned,summary.coverage,marker='o')
for _,r in summary.iterrows(): plt.annotate(f'{r.threshold:.3f}',(r.print_loss_per_annotation_cleaned,r.coverage))
plt.xlabel('Print lost per annotation pixel cleaned'); plt.ylabel('Eligible CLEAN coverage'); plt.title('A5 safety / coverage frontier'); plt.tight_layout(); plt.savefig(VISUAL/'safety_coverage_frontier.png',dpi=180); plt.close()

env={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; (OUT/'software_environment.json').write_text(json.dumps(env,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook89-a5-visual-audit','development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'source_run':str(SOURCE),'source_predictions_sha256':sha256_file(PRED),'thresholds':THRESHOLDS,'review_radius':REVIEW_RADIUS,'pages':int(full.page.nunique()),'rows':int(len(full)),'component_crops':int(len(crop_index)),'software':env}; (OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2))
text='NOTEBOOK 89 A5 VISUAL AUDIT\n\nSource: '+str(SOURCE)+'Retrained: False\nRefitted: False\nRecalibrated: False\n\n'+summary.to_string(index=False)+'\n\nGreen=annotation cleaned; red=print lost; purple=truth uncertain.\n'
(OUT/'executive_summary.txt').write_text(text)

curated=[OUT/'executive_summary.txt',OUT/'experiment_identity.json',OUT/'software_environment.json',TABLES/'threshold_summary.csv',TABLES/'results_by_page.csv',TABLES/'results_by_component.csv',TABLES/'component_crop_index.csv',VISUAL/'safety_coverage_frontier.png']+[TABLES/f'{x}_review_queue.csv' for x in queues]+[VISUAL/f'{x}_contact_sheet.png' for x in queues]
for p in curated:
    if p.exists():
        dst=UPLOAD/p.relative_to(OUT); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 89 review bundle. Start with executive_summary.txt and visual_audit contact sheets. Full-resolution crops remain in the run directory. Green=good cleanup; red=print loss; purple=uncertain.\n')
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=UPLOAD/'notebook89_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
print(text); print('Upload bundle:',zip_path)


## Inspect first

- `visual_audit/catastrophic_contact_sheet.png`
- `visual_audit/false_heavy_contact_sheet.png`
- `visual_audit/beneficial_contact_sheet.png`
- `visual_audit/threshold_boundary_contact_sheet.png`
- `tables/*_review_queue.csv`
- `crops/<page>/<page>_component_<id>.png`
- `whole_pages/<page>/whole_page_review.png`
- `upload/notebook89_upload_bundle.zip`
